## Import dan Direktori Project

In [1]:
from pathlib import Path
import pandas as pd
import json
import sys
import importlib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

direktori_aktif = Path.cwd().resolve()

if direktori_aktif.name == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_src = direktori_project / "src"
direktori_models = direktori_project / "models"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_samples_metadata = direktori_project / "data" / "samples_metadata"

direktori_src.mkdir(parents=True, exist_ok=True)
direktori_outputs.mkdir(parents=True, exist_ok=True)

if str(direktori_src) not in sys.path:
    sys.path.append(str(direktori_src))

print("Direktori aktif notebook:", direktori_aktif)
print("Direktori project:", direktori_project)
print("Folder src:", direktori_src)
print("Folder models:", direktori_models)
print("Folder outputs:", direktori_outputs)

Direktori aktif notebook: C:\Users\ASUS\PHISHING\notebooks
Direktori project: C:\Users\ASUS\PHISHING
Folder src: C:\Users\ASUS\PHISHING\src
Folder models: C:\Users\ASUS\PHISHING\models
Folder outputs: C:\Users\ASUS\PHISHING\reports\outputs


## Validasi File Penting

In [2]:
lokasi_model_v2 = direktori_models / "model_terbaik_intelligence_v2.pkl"
lokasi_fitur_v2 = direktori_outputs / "daftar_fitur_intelligence_v2.json"
lokasi_url_intelligence = direktori_src / "url_intelligence.py"
lokasi_file_static_analyzer = direktori_src / "file_static_analyzer.py"

daftar_file_wajib_step9 = [
    lokasi_model_v2,
    lokasi_fitur_v2,
    lokasi_url_intelligence,
    lokasi_file_static_analyzer,
    direktori_project / "data" / "intelligence" / "official_domains_global.csv",
    direktori_project / "data" / "intelligence" / "brand_keywords_global.csv",
    direktori_project / "data" / "intelligence" / "suspicious_keywords_global.csv"
]

hasil_validasi_awal_step9 = []

for lokasi_file in daftar_file_wajib_step9:
    hasil_validasi_awal_step9.append({
        "nama_file": lokasi_file.name,
        "lokasi": str(lokasi_file),
        "tersedia": lokasi_file.exists(),
        "ukuran_kb": round(lokasi_file.stat().st_size / 1024, 2) if lokasi_file.exists() else 0
    })

data_validasi_awal_step9 = pd.DataFrame(hasil_validasi_awal_step9)

if not data_validasi_awal_step9["tersedia"].all():
    display(data_validasi_awal_step9)
    raise FileNotFoundError("Ada file wajib STEP 9 yang belum tersedia.")

print("Semua file wajib STEP 9 tersedia.")
data_validasi_awal_step9

Semua file wajib STEP 9 tersedia.


,nama_file,lokasi,tersedia,ukuran_kb
0,model_terbaik_intelligence_v2.pkl,C:\Users\ASUS\PHISHING\models\model_terbaik_in...,True,49114.43
1,daftar_fitur_intelligence_v2.json,C:\Users\ASUS\PHISHING\reports\outputs\daftar_...,True,1.02
2,url_intelligence.py,C:\Users\ASUS\PHISHING\src\url_intelligence.py,True,15.60
3,file_static_analyzer.py,C:\Users\ASUS\PHISHING\src\file_static_analyze...,True,16.67
4,official_domains_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\offic...,True,4.31
5,brand_keywords_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\brand...,True,1.43
6,suspicious_keywords_global.csv,C:\Users\ASUS\PHISHING\data\intelligence\suspi...,True,1.78


## Membuat phishrisk_engine_v3.py

In [3]:
kode_phishrisk_engine_v3 = r'''
from pathlib import Path
from urllib.parse import urlparse
import hashlib
import json
import re
import sys

import joblib
import pandas as pd


DIREKTORI_SRC = Path(__file__).resolve().parent

if str(DIREKTORI_SRC) not in sys.path:
    sys.path.append(str(DIREKTORI_SRC))

import url_intelligence
import file_static_analyzer


FITUR_INTELLIGENCE_V2 = [
    "is_official_domain",
    "brand_keyword_detected",
    "brand_but_not_official",
    "suspicious_keyword_count",
    "suspicious_keyword_score",
    "lookalike_brand_detected",
    "lookalike_score",
    "uses_punycode",
    "uses_digit_substitution",
    "hyphen_count"
]


def ambil_domain_dari_url(url):
    url = str(url).strip()
    hasil_parse = urlparse(url)

    if hasil_parse.netloc:
        return hasil_parse.netloc.lower().split("@")[-1].split(":")[0]

    hasil_parse = urlparse("http://" + url)
    return hasil_parse.netloc.lower().split("@")[-1].split(":")[0]


def cek_domain_ip(domain):
    pola_ip = r"^\d{1,3}(\.\d{1,3}){3}$"
    return int(bool(re.match(pola_ip, str(domain).strip())))


def ambil_tld(domain):
    bagian = [item for item in str(domain).lower().split(".") if item]

    if not bagian:
        return "tidak_diketahui"

    return bagian[-1]


def hitung_subdomain(domain):
    domain = str(domain).lower().strip()
    bagian = [item for item in domain.split(".") if item]

    if cek_domain_ip(domain):
        return 0

    if len(bagian) <= 2:
        return 0

    return len(bagian) - 2


def hapus_skema_url(url):
    return re.sub(r"^https?://", "", str(url).strip(), flags=re.IGNORECASE)


def hapus_www_awal(teks):
    return re.sub(r"^www\.", "", str(teks).strip(), flags=re.IGNORECASE)


def hitung_obfuscation(url):
    pola = r"%[0-9a-fA-F]{2}"
    return len(re.findall(pola, str(url)))


def kategori_dari_skor(skor):
    skor = float(skor)

    if skor < 25:
        return "Rendah"

    if skor < 50:
        return "Sedang"

    if skor < 75:
        return "Tinggi"

    return "Sangat Tinggi"


def ekstrak_url_dari_teks(teks):
    teks = str(teks)
    pola_url = r"https?://[^\s<>'\"\\)\]\}]+"
    daftar_url = re.findall(pola_url, teks, flags=re.IGNORECASE)

    daftar_bersih = []

    for url in daftar_url:
        url = url.strip().rstrip(".,;:")
        if url not in daftar_bersih:
            daftar_bersih.append(url)

    return daftar_bersih


class PhishRiskEngineV3:
    def __init__(self, direktori_project):
        self.direktori_project = Path(direktori_project)

        self.lokasi_model = self.direktori_project / "models" / "model_terbaik_intelligence_v2.pkl"
        self.lokasi_fitur = self.direktori_project / "reports" / "outputs" / "daftar_fitur_intelligence_v2.json"

        if not self.lokasi_model.exists():
            raise FileNotFoundError(f"Model Intelligence V2 tidak ditemukan: {self.lokasi_model}")

        if not self.lokasi_fitur.exists():
            raise FileNotFoundError(f"Daftar fitur Intelligence V2 tidak ditemukan: {self.lokasi_fitur}")

        self.model = joblib.load(self.lokasi_model)

        with open(self.lokasi_fitur, "r", encoding="utf-8") as file:
            self.daftar_fitur_model = json.load(file)

        self.data_domain_resmi, self.data_brand_keyword, self.data_suspicious_keyword = url_intelligence.muat_data_intelligence(
            self.direktori_project
        )

    def ekstrak_fitur_url_manual(self, url):
        url = str(url).strip()
        domain = ambil_domain_dari_url(url)
        tld = ambil_tld(domain)

        url_tanpa_skema = hapus_skema_url(url)
        url_untuk_hitung = hapus_www_awal(url_tanpa_skema)

        panjang_url = len(url)
        panjang_domain = len(domain)

        jumlah_obfuscation = hitung_obfuscation(url)
        jumlah_huruf = sum(karakter.isalpha() for karakter in url_untuk_hitung)
        jumlah_angka = sum(karakter.isdigit() for karakter in url_untuk_hitung)

        jumlah_sama_dengan = url_untuk_hitung.count("=")
        jumlah_tanda_tanya = url_untuk_hitung.count("?")
        jumlah_ampersand = url_untuk_hitung.count("&")

        karakter_khusus = re.findall(r"[^a-zA-Z0-9]", url_untuk_hitung)
        jumlah_karakter_khusus = len(karakter_khusus)

        fitur = {
            "URLLength": panjang_url,
            "DomainLength": panjang_domain,
            "IsDomainIP": cek_domain_ip(domain),
            "TLDLength": len(tld),
            "NoOfSubDomain": hitung_subdomain(domain),
            "HasObfuscation": int(jumlah_obfuscation > 0),
            "NoOfObfuscatedChar": jumlah_obfuscation * 3,
            "ObfuscationRatio": (jumlah_obfuscation * 3 / panjang_url) if panjang_url > 0 else 0,
            "NoOfLettersInURL": jumlah_huruf,
            "LetterRatioInURL": (jumlah_huruf / panjang_url) if panjang_url > 0 else 0,
            "NoOfDegitsInURL": jumlah_angka,
            "DegitRatioInURL": (jumlah_angka / panjang_url) if panjang_url > 0 else 0,
            "NoOfEqualsInURL": jumlah_sama_dengan,
            "NoOfQMarkInURL": jumlah_tanda_tanya,
            "NoOfAmpersandInURL": jumlah_ampersand,
            "NoOfOtherSpecialCharsInURL": jumlah_karakter_khusus,
            "SpacialCharRatioInURL": (jumlah_karakter_khusus / panjang_url) if panjang_url > 0 else 0,
            "IsHTTPS": int(url.lower().startswith("https://"))
        }

        for nama_fitur in self.daftar_fitur_model:
            if nama_fitur.startswith("TLD_"):
                nama_tld = nama_fitur.replace("TLD_", "")
                fitur[nama_fitur] = int(tld == nama_tld)

        fitur_tld_tersedia = [fitur for fitur in self.daftar_fitur_model if fitur.startswith("TLD_")]
        daftar_tld_model = [fitur.replace("TLD_", "") for fitur in fitur_tld_tersedia if fitur != "TLD_lainnya"]

        if f"TLD_{tld}" not in fitur_tld_tersedia and "TLD_lainnya" in self.daftar_fitur_model:
            fitur["TLD_lainnya"] = 1

        return fitur, domain, tld

    def ekstrak_fitur_url_v3(self, url):
        fitur_manual, domain, tld = self.ekstrak_fitur_url_manual(url)

        sinyal_intelligence = url_intelligence.analisis_url_intelligence(
            url,
            self.data_domain_resmi,
            self.data_brand_keyword,
            self.data_suspicious_keyword
        )

        fitur_intelligence = {}

        for nama_fitur in FITUR_INTELLIGENCE_V2:
            fitur_intelligence[nama_fitur] = sinyal_intelligence.get(nama_fitur, 0)

        fitur_gabungan = {}
        fitur_gabungan.update(fitur_manual)
        fitur_gabungan.update(fitur_intelligence)

        data_fitur = pd.DataFrame([fitur_gabungan])
        data_fitur = data_fitur.reindex(columns=self.daftar_fitur_model, fill_value=0)
        data_fitur = data_fitur.fillna(0)

        return data_fitur, domain, tld, sinyal_intelligence

    def kalibrasi_skor_url(self, skor_model, sinyal_intelligence):
        skor_model = float(skor_model)
        status = sinyal_intelligence.get("intelligence_status", "")

        if status == "resmi_terlihat_aman":
            if skor_model >= 60:
                return 35.0
            if skor_model >= 50:
                return 28.0
            return min(skor_model, 24.0)

        if status == "resmi_perlu_tinjauan":
            return max(min(skor_model, 60.0), 35.0)

        if status == "tiruan_brand_berisiko":
            return max(skor_model, 85.0)

        if status == "domain_mirip_brand_berisiko":
            return max(skor_model, 85.0)

        if status == "domain_mirip_brand":
            return max(skor_model, 75.0)

        if status == "kata_mencurigakan_tinggi":
            return max(skor_model, 75.0)

        if status == "punycode_perlu_tinjauan":
            return max(skor_model, 65.0)

        if status == "domain_ip_perlu_tinjauan":
            return max(skor_model, 65.0)

        if status == "brand_tidak_resmi_perlu_tinjauan":
            return max(skor_model, 60.0)

        if status == "kata_mencurigakan_perlu_tinjauan":
            return max(skor_model, 45.0)

        return skor_model

    def tentukan_hasil_akhir_url(self, skor_final, sinyal_intelligence):
        status = sinyal_intelligence.get("intelligence_status", "")
        skor_final = float(skor_final)

        if status == "resmi_terlihat_aman":
            if skor_final < 25:
                return "Terlihat Aman"
            return "Perlu Tinjauan"

        if status == "resmi_perlu_tinjauan":
            return "Perlu Tinjauan"

        if status in [
            "tiruan_brand_berisiko",
            "domain_mirip_brand_berisiko",
            "kata_mencurigakan_tinggi"
        ]:
            return "Berisiko"

        if status in [
            "domain_mirip_brand",
            "punycode_perlu_tinjauan",
            "domain_ip_perlu_tinjauan",
            "brand_tidak_resmi_perlu_tinjauan",
            "kata_mencurigakan_perlu_tinjauan"
        ]:
            if skor_final >= 75:
                return "Berisiko"
            return "Perlu Tinjauan"

        if skor_final < 25:
            return "Terlihat Aman"

        if skor_final < 50:
            return "Perlu Tinjauan"

        return "Berisiko"

    def buat_rekomendasi_url(self, hasil_akhir, skor_model, skor_final, sinyal_intelligence):
        status = sinyal_intelligence.get("intelligence_status", "")

        if status == "resmi_terlihat_aman" and hasil_akhir == "Terlihat Aman":
            return "Alamat cocok dengan daftar domain resmi dan tidak menunjukkan sinyal kuat yang mencurigakan. Tetap pastikan alamat diketik langsung dari sumber resmi."

        if status == "resmi_terlihat_aman" and hasil_akhir == "Perlu Tinjauan":
            return "Alamat cocok dengan daftar domain resmi, tetapi pola URL perlu diperiksa. Jangan langsung dianggap phishing, namun tetap pastikan alamat berasal dari kanal resmi."

        if status == "resmi_perlu_tinjauan":
            return "Alamat berada pada domain resmi, tetapi memiliki pola yang perlu diperiksa. Pastikan tidak ada path, parameter, atau redirect mencurigakan."

        if status in ["tiruan_brand_berisiko", "domain_mirip_brand_berisiko", "domain_mirip_brand"]:
            return "Alamat terindikasi meniru brand atau domain resmi. Jangan digunakan untuk login, transaksi, atau memasukkan data pribadi."

        if hasil_akhir == "Terlihat Aman":
            return "Alamat terlihat aman berdasarkan model dan pemeriksaan intelligence. Tetap pastikan alamat berasal dari sumber tepercaya."

        if hasil_akhir == "Perlu Tinjauan":
            return "Alamat perlu diperiksa manual. Jangan langsung memasukkan kata sandi, OTP, data pribadi, atau informasi pembayaran."

        return "Alamat berisiko. Jangan dibuka, jangan diisi, dan laporkan jika berasal dari pesan mencurigakan."

    def analisis_url(self, url):
        data_fitur, domain, tld, sinyal = self.ekstrak_fitur_url_v3(url)

        probabilitas_phishing = float(self.model.predict_proba(data_fitur)[0][1])
        skor_model = round(probabilitas_phishing * 100, 2)

        prediksi_model = int(probabilitas_phishing >= 0.5)
        label_model = "Phishing" if prediksi_model == 1 else "Legitimate"

        skor_final = round(self.kalibrasi_skor_url(skor_model, sinyal), 2)
        kategori_risiko = kategori_dari_skor(skor_final)
        hasil_akhir = self.tentukan_hasil_akhir_url(skor_final, sinyal)
        rekomendasi = self.buat_rekomendasi_url(hasil_akhir, skor_model, skor_final, sinyal)

        hasil = {
            "url": str(url),
            "domain": domain,
            "tld": tld,
            "probabilitas_model": round(probabilitas_phishing, 6),
            "skor_model": skor_model,
            "label_model": label_model,
            "skor_final": skor_final,
            "kategori_risiko": kategori_risiko,
            "hasil_akhir": hasil_akhir,
            "rekomendasi": rekomendasi,
            "intelligence_status": sinyal.get("intelligence_status", ""),
            "intelligence_reason": sinyal.get("intelligence_reason", ""),
            "is_official_domain": sinyal.get("is_official_domain", 0),
            "official_brand": sinyal.get("official_brand", ""),
            "official_domain": sinyal.get("official_domain", ""),
            "brand_detected": sinyal.get("brand_detected", ""),
            "brand_but_not_official": sinyal.get("brand_but_not_official", 0),
            "suspicious_keywords": sinyal.get("suspicious_keywords", ""),
            "suspicious_keyword_score": sinyal.get("suspicious_keyword_score", 0),
            "lookalike_brand_detected": sinyal.get("lookalike_brand_detected", 0),
            "lookalike_brand": sinyal.get("lookalike_brand", ""),
            "lookalike_score": sinyal.get("lookalike_score", 0),
            "uses_punycode": sinyal.get("uses_punycode", 0),
            "uses_digit_substitution": sinyal.get("uses_digit_substitution", 0),
            "hyphen_count": sinyal.get("hyphen_count", 0)
        }

        return hasil

    def analisis_banyak_url(self, daftar_url):
        hasil = []

        for url in daftar_url:
            hasil.append(self.analisis_url(url))

        return pd.DataFrame(hasil)

    def analisis_file(self, lokasi_file):
        hasil_file_statis, hasil_url_intelligence = file_static_analyzer.analisis_file_statis(
            lokasi_file,
            self.direktori_project
        )

        daftar_url = []

        url_terdeteksi = hasil_file_statis.get("url_terdeteksi", "")

        if isinstance(url_terdeteksi, str) and url_terdeteksi.strip():
            daftar_url = [url.strip() for url in url_terdeteksi.split(" | ") if url.strip()]

        if len(daftar_url) > 0:
            data_url_v3 = self.analisis_banyak_url(daftar_url)
        else:
            data_url_v3 = pd.DataFrame()

        jumlah_url_berisiko_v3 = 0
        jumlah_url_tinjauan_v3 = 0

        if not data_url_v3.empty:
            jumlah_url_berisiko_v3 = int((data_url_v3["hasil_akhir"] == "Berisiko").sum())
            jumlah_url_tinjauan_v3 = int((data_url_v3["hasil_akhir"] == "Perlu Tinjauan").sum())

        skor_file_awal = float(hasil_file_statis.get("skor_risiko_file", 0))
        skor_final = skor_file_awal

        if jumlah_url_berisiko_v3 > 0:
            skor_final = max(skor_final, 80)

        if jumlah_url_tinjauan_v3 > 0:
            skor_final = max(skor_final, 55)

        if hasil_file_statis.get("jumlah_izin_apk_berisiko", 0) > 0:
            skor_final = max(skor_final, 85)

        if hasil_file_statis.get("pdf_javascript", 0) == 1:
            skor_final = max(skor_final, 85)

        if hasil_file_statis.get("pdf_open_action", 0) == 1 or hasil_file_statis.get("pdf_launch_action", 0) == 1:
            skor_final = max(skor_final, 85)

        if hasil_file_statis.get("jumlah_file_berbahaya_dalam_arsip", 0) > 0:
            skor_final = max(skor_final, 85)

        skor_final = int(min(100, max(0, skor_final)))
        kategori_final = kategori_dari_skor(skor_final)

        if skor_final < 25:
            hasil_akhir_file = "Terlihat Aman"
        elif skor_final < 50:
            hasil_akhir_file = "Perlu Tinjauan"
        else:
            hasil_akhir_file = "Berisiko"

        hasil_file_statis["jumlah_url_berisiko_v3"] = jumlah_url_berisiko_v3
        hasil_file_statis["jumlah_url_perlu_tinjauan_v3"] = jumlah_url_tinjauan_v3
        hasil_file_statis["skor_final_file_v3"] = skor_final
        hasil_file_statis["kategori_final_file_v3"] = kategori_final
        hasil_file_statis["hasil_akhir_file_v3"] = hasil_akhir_file

        if hasil_akhir_file == "Terlihat Aman":
            hasil_file_statis["rekomendasi_final_file_v3"] = "File terlihat rendah risiko berdasarkan pemeriksaan statis. Tetap buka hanya jika sumbernya tepercaya."
        elif hasil_akhir_file == "Perlu Tinjauan":
            hasil_file_statis["rekomendasi_final_file_v3"] = "File perlu diperiksa manual sebelum dibuka, terutama jika berasal dari pesan, email, atau sumber tidak dikenal."
        else:
            hasil_file_statis["rekomendasi_final_file_v3"] = "File berisiko. Jangan dibuka atau dijalankan sebelum diperiksa di lingkungan aman."

        return hasil_file_statis, data_url_v3

    def analisis_banyak_file(self, daftar_lokasi_file):
        hasil_file_semua = []
        hasil_url_semua = []

        for lokasi_file in daftar_lokasi_file:
            hasil_file, data_url_v3 = self.analisis_file(lokasi_file)
            hasil_file_semua.append(hasil_file)

            if not data_url_v3.empty:
                data_url_v3 = data_url_v3.copy()
                data_url_v3.insert(0, "nama_file_sumber", Path(lokasi_file).name)
                hasil_url_semua.append(data_url_v3)

        data_file = pd.DataFrame(hasil_file_semua)

        if hasil_url_semua:
            data_url = pd.concat(hasil_url_semua, ignore_index=True)
        else:
            data_url = pd.DataFrame()

        return data_file, data_url


def buat_engine(direktori_project):
    return PhishRiskEngineV3(direktori_project)


def analisis_url_v3(url, direktori_project):
    engine = PhishRiskEngineV3(direktori_project)
    return engine.analisis_url(url)


def analisis_banyak_url_v3(daftar_url, direktori_project):
    engine = PhishRiskEngineV3(direktori_project)
    return engine.analisis_banyak_url(daftar_url)


def analisis_file_v3(lokasi_file, direktori_project):
    engine = PhishRiskEngineV3(direktori_project)
    return engine.analisis_file(lokasi_file)


def analisis_banyak_file_v3(daftar_lokasi_file, direktori_project):
    engine = PhishRiskEngineV3(direktori_project)
    return engine.analisis_banyak_file(daftar_lokasi_file)
'''

lokasi_engine_v3 = direktori_src / "phishrisk_engine_v3.py"

with open(lokasi_engine_v3, "w", encoding="utf-8") as file:
    file.write(kode_phishrisk_engine_v3)

print("PhishRisk Engine V3 berhasil dibuat:")
print(lokasi_engine_v3)

PhishRisk Engine V3 berhasil dibuat:
C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py


## Uji Engine V3 untuk URL

In [4]:
import phishrisk_engine_v3
importlib.reload(phishrisk_engine_v3)

engine_v3 = phishrisk_engine_v3.buat_engine(direktori_project)

daftar_url_uji_v3 = [
    "https://praktikum.gunadarma.ac.id",
    "https://baak.gunadarma.ac.id",
    "https://www.bca.co.id",
    "https://www.shopee.co.id",
    "https://www.microsoft.com",
    "http://rricrosoft.com",
    "http://rnicrosoft.com",
    "http://micros0ft-login-update.test",
    "http://bca-login-update.test",
    "http://paypal-verify-account.test",
    "http://praktikum-gunadarma-login-update.test",
    "http://155.94.163.206/ai/?authenticated=true&account=login",
    "https://xn--micrsoft-q4a.test",
    "http://c1mb.test",
    "http://maybanksecure.test"
]

data_hasil_url_v3 = engine_v3.analisis_banyak_url(daftar_url_uji_v3)

kolom_tampilan_url_v3 = [
    "url",
    "domain",
    "label_model",
    "skor_model",
    "skor_final",
    "kategori_risiko",
    "hasil_akhir",
    "intelligence_status",
    "official_brand",
    "brand_detected",
    "suspicious_keywords",
    "lookalike_brand",
    "lookalike_score",
    "rekomendasi"
]

data_hasil_url_v3[kolom_tampilan_url_v3]

,url,domain,label_model,skor_model,skor_final,kategori_risiko,hasil_akhir,intelligence_status,official_brand,brand_detected,suspicious_keywords,lookalike_brand,lookalike_score,rekomendasi
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,Legitimate,20.40,20.40,Rendah,Terlihat Aman,resmi_terlihat_aman,Gunadarma,Gunadarma,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
1,https://baak.gunadarma.ac.id,baak.gunadarma.ac.id,Legitimate,31.60,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,Gunadarma,Gunadarma,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
2,https://www.bca.co.id,www.bca.co.id,Legitimate,4.05,4.05,Rendah,Terlihat Aman,resmi_terlihat_aman,BCA,BCA,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
3,https://www.shopee.co.id,www.shopee.co.id,Legitimate,9.22,9.22,Rendah,Terlihat Aman,resmi_terlihat_aman,Shopee,Shopee,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
4,https://www.microsoft.com,www.microsoft.com,Legitimate,48.00,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,Microsoft,Microsoft,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
5,http://rricrosoft.com,rricrosoft.com,Phishing,99.60,99.60,Sangat Tinggi,Berisiko,domain_mirip_brand,,,,Microsoft,0.8421,Alamat terindikasi meniru brand atau domain re...
6,http://rnicrosoft.com,rnicrosoft.com,Phishing,99.60,99.60,Sangat Tinggi,Berisiko,domain_mirip_brand,,,,Microsoft,0.8421,Alamat terindikasi meniru brand atau domain re...
7,http://micros0ft-login-update.test,micros0ft-login-update.test,Phishing,98.80,98.80,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,,Microsoft,"login, update",Microsoft,1.0000,Alamat terindikasi meniru brand atau domain re...
8,http://bca-login-update.test,bca-login-update.test,Phishing,99.60,99.60,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,,BCA,"login, update",BCA,1.0000,Alamat terindikasi meniru brand atau domain re...
9,http://paypal-verify-account.test,paypal-verify-account.test,Phishing,100.00,100.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,,PayPal,"account, verify",PayPal,1.0000,Alamat terindikasi meniru brand atau domain re...


## Uji Engine V3 untuk File

In [5]:
direktori_sample_file = direktori_samples_metadata / "file_samples"

if not direktori_sample_file.exists():
    raise FileNotFoundError("Folder file sample belum ditemukan. Jalankan dulu cell pembuatan sample dari notebook 06.")

daftar_file_sample_v3 = sorted([
    lokasi_file
    for lokasi_file in direktori_sample_file.iterdir()
    if lokasi_file.is_file()
])

print("Jumlah file sample:", len(daftar_file_sample_v3))

data_hasil_file_v3, data_hasil_url_dalam_file_v3 = engine_v3.analisis_banyak_file(daftar_file_sample_v3)

kolom_tampilan_file_v3 = [
    "nama_file",
    "ekstensi",
    "risiko_awal_file",
    "jumlah_url",
    "jumlah_url_berisiko_v3",
    "jumlah_url_perlu_tinjauan_v3",
    "jumlah_kata_mencurigakan",
    "jumlah_izin_apk_berisiko",
    "skor_risiko_file",
    "skor_final_file_v3",
    "kategori_final_file_v3",
    "hasil_akhir_file_v3",
    "alasan_file",
    "rekomendasi_final_file_v3"
]

data_hasil_file_v3[kolom_tampilan_file_v3]

Jumlah file sample: 7


,nama_file,ekstensi,risiko_awal_file,jumlah_url,jumlah_url_berisiko_v3,jumlah_url_perlu_tinjauan_v3,jumlah_kata_mencurigakan,jumlah_izin_apk_berisiko,skor_risiko_file,skor_final_file_v3,kategori_final_file_v3,hasil_akhir_file_v3,alasan_file,rekomendasi_final_file_v3
0,contoh_aplikasi_dummy.apk,.apk,tinggi,1,1,0,2,3,100,100,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
1,contoh_arsip_mencurigakan.zip,.zip,sedang,2,2,0,4,0,100,100,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
2,contoh_catatan_aman.txt,.txt,rendah,1,0,0,0,0,12,12,Rendah,Terlihat Aman,File mengandung URL.,File terlihat rendah risiko berdasarkan pemeri...
3,contoh_dokumen_link.docx,.docx,sedang,2,2,0,4,0,93,93,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
4,contoh_halaman_login.html,.html,sedang,1,1,0,5,0,74,80,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
5,contoh_pdf_link.pdf,.pdf,sedang,1,1,0,2,0,100,100,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
6,contoh_pesan_mencurigakan.txt,.txt,rendah,1,1,0,5,0,55,80,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...


## Lihat URL yang Ditemukan di Dalam File

In [6]:
if data_hasil_url_dalam_file_v3.empty:
    print("Tidak ada URL yang ditemukan di dalam file.")
else:
    kolom_url_dalam_file = [
        "nama_file_sumber",
        "url",
        "domain",
        "label_model",
        "skor_model",
        "skor_final",
        "hasil_akhir",
        "intelligence_status",
        "brand_detected",
        "suspicious_keywords",
        "lookalike_brand",
        "rekomendasi"
    ]

    display(data_hasil_url_dalam_file_v3[kolom_url_dalam_file].head(50))

,nama_file_sumber,url,domain,label_model,skor_model,skor_final,hasil_akhir,intelligence_status,brand_detected,suspicious_keywords,lookalike_brand,rekomendasi
0,contoh_aplikasi_dummy.apk,http://ovo-login-update.test,ovo-login-update.test,Phishing,99.6,99.6,Berisiko,tiruan_brand_berisiko,OVO,"login, update",OVO,Alamat terindikasi meniru brand atau domain re...
1,contoh_arsip_mencurigakan.zip,http://paypal-verify-account.testPK     CX,paypal-verify-account.testpk     cx,Phishing,97.2,97.2,Berisiko,tiruan_brand_berisiko,PayPal,"account, verify",PayPal,Alamat terindikasi meniru brand atau domain re...
2,contoh_arsip_mencurigakan.zip,http://paypal-verify-account.test,paypal-verify-account.test,Phishing,100.0,100.0,Berisiko,tiruan_brand_berisiko,PayPal,"account, verify",PayPal,Alamat terindikasi meniru brand atau domain re...
3,contoh_catatan_aman.txt,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,Legitimate,20.4,20.4,Terlihat Aman,resmi_terlihat_aman,Gunadarma,,,Alamat cocok dengan daftar domain resmi dan ti...
4,contoh_dokumen_link.docx,http://shopee-login-update.test,shopee-login-update.test,Phishing,100.0,100.0,Berisiko,tiruan_brand_berisiko,Shopee,"login, update",Shopee,Alamat terindikasi meniru brand atau domain re...
5,contoh_dokumen_link.docx,http://bca-login-update.test,bca-login-update.test,Phishing,99.6,99.6,Berisiko,tiruan_brand_berisiko,BCA,"login, update",BCA,Alamat terindikasi meniru brand atau domain re...
6,contoh_halaman_login.html,http://paypal-verify-account.test/login,paypal-verify-account.test,Phishing,100.0,100.0,Berisiko,tiruan_brand_berisiko,PayPal,"account, login, verify",PayPal,Alamat terindikasi meniru brand atau domain re...
7,contoh_pdf_link.pdf,https://micros0ft-login-update.test,micros0ft-login-update.test,Phishing,93.2,93.2,Berisiko,tiruan_brand_berisiko,Microsoft,"login, update",Microsoft,Alamat terindikasi meniru brand atau domain re...
8,contoh_pesan_mencurigakan.txt,http://bca-login-update.test,bca-login-update.test,Phishing,99.6,99.6,Berisiko,tiruan_brand_berisiko,BCA,"login, update",BCA,Alamat terindikasi meniru brand atau domain re...


## Simpan Hasil

In [7]:
lokasi_hasil_url_v3 = direktori_outputs / "hasil_uji_phishrisk_engine_v3_url.csv"
lokasi_hasil_file_v3 = direktori_outputs / "hasil_uji_phishrisk_engine_v3_file.csv"
lokasi_hasil_url_dalam_file_v3 = direktori_outputs / "hasil_uji_phishrisk_engine_v3_url_dalam_file.csv"

data_hasil_url_v3.to_csv(
    lokasi_hasil_url_v3,
    index=False,
    encoding="utf-8"
)

data_hasil_file_v3.to_csv(
    lokasi_hasil_file_v3,
    index=False,
    encoding="utf-8"
)

data_hasil_url_dalam_file_v3.to_csv(
    lokasi_hasil_url_dalam_file_v3,
    index=False,
    encoding="utf-8"
)

print("Hasil STEP 9 disimpan:")
print(lokasi_hasil_url_v3)
print(lokasi_hasil_file_v3)
print(lokasi_hasil_url_dalam_file_v3)

Hasil STEP 9 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_phishrisk_engine_v3_url.csv
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_phishrisk_engine_v3_file.csv
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_phishrisk_engine_v3_url_dalam_file.csv


## Metadata Engine V3

In [ ]:
metadata_engine_v3 = {
    "nama_program": "PhishRisk Intelligence Engine",
    "versi_engine": "V3",
    "step": "STEP 9",
    "fungsi_utama": [
        "analisis_url",
        "analisis_banyak_url",
        "analisis_file",
        "analisis_banyak_file",
        "analisis_url_di_dalam_file"
    ],
    "model_digunakan": str(lokasi_model_v2),
    "fitur_digunakan": str(lokasi_fitur_v2),
    "modul_engine": str(lokasi_engine_v3),
    "modul_url_intelligence": str(lokasi_url_intelligence),
    "modul_file_static_analyzer": str(lokasi_file_static_analyzer),
    "output_url_v3": str(lokasi_hasil_url_v3),
    "output_file_v3": str(lokasi_hasil_file_v3),
    "output_url_dalam_file_v3": str(lokasi_hasil_url_dalam_file_v3),
    "catatan": "Engine V3 memakai model Intelligence V2 dan kalibrasi final untuk URL resmi, tiruan brand, lookalike domain, dan file statis."
}

lokasi_metadata_engine_v3 = direktori_outputs / "metadata_phishrisk_engine_v3.json"

with open(lokasi_metadata_engine_v3, "w", encoding="utf-8") as file:
    json.dump(metadata_engine_v3, file, indent=4, ensure_ascii=False)

print("Metadata Engine V3 disimpan:")
print(lokasi_metadata_engine_v3)

metadata_engine_v3

Metadata Engine V3 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\metadata_phishrisk_engine_v3.json


{'nama_program': 'PhishRisk Intelligence Engine',
 'versi_engine': 'V3',
 'step': 'STEP 9',
 'fungsi_utama': ['analisis_url',
  'analisis_banyak_url',
  'analisis_file',
  'analisis_banyak_file',
  'analisis_url_di_dalam_file'],
 'model_digunakan': 'C:\\Users\\ASUS\\PHISHING\\models\\model_terbaik_intelligence_v2.pkl',
 'fitur_digunakan': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\daftar_fitur_intelligence_v2.json',
 'modul_engine': 'C:\\Users\\ASUS\\PHISHING\\src\\phishrisk_engine_v3.py',
 'modul_url_intelligence': 'C:\\Users\\ASUS\\PHISHING\\src\\url_intelligence.py',
 'modul_file_static_analyzer': 'C:\\Users\\ASUS\\PHISHING\\src\\file_static_analyzer.py',
 'output_url_v3': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\hasil_uji_phishrisk_engine_v3_url.csv',
 'output_file_v3': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\hasil_uji_phishrisk_engine_v3_file.csv',
 'output_url_dalam_file_v3': 'C:\\Users\\ASUS\\PHISHING\\reports\\outputs\\hasil_uji_phishrisk_engine_v3_url_dalam_file.

## Validasi Final

In [9]:
daftar_file_validasi_step9 = [
    lokasi_engine_v3,
    lokasi_hasil_url_v3,
    lokasi_hasil_file_v3,
    lokasi_hasil_url_dalam_file_v3,
    lokasi_metadata_engine_v3
]

hasil_validasi_step9 = []

for lokasi_file in daftar_file_validasi_step9:
    hasil_validasi_step9.append({
        "nama_file": lokasi_file.name,
        "lokasi": str(lokasi_file),
        "tersedia": lokasi_file.exists(),
        "ukuran_kb": round(lokasi_file.stat().st_size / 1024, 2) if lokasi_file.exists() else 0
    })

data_validasi_step9 = pd.DataFrame(hasil_validasi_step9)

lokasi_validasi_step9 = direktori_outputs / "validasi_step9_phishrisk_engine_v3.csv"

data_validasi_step9.to_csv(
    lokasi_validasi_step9,
    index=False,
    encoding="utf-8"
)

print("Validasi STEP 9 disimpan:")
print(lokasi_validasi_step9)

data_validasi_step9

Validasi STEP 9 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\validasi_step9_phishrisk_engine_v3.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,phishrisk_engine_v3.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py,True,17.66
1,hasil_uji_phishrisk_engine_v3_url.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,6.55
2,hasil_uji_phishrisk_engine_v3_file.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,5.93
3,hasil_uji_phishrisk_engine_v3_url_dalam_file.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,5.12
4,metadata_phishrisk_engine_v3.json,C:\Users\ASUS\PHISHING\reports\outputs\metadat...,True,1.22


## Ringkasan

In [10]:
print("RINGKASAN STEP INI")
print("=" * 60)

print("Engine final:", lokasi_engine_v3)
print("Model digunakan:", lokasi_model_v2)
print("Fitur digunakan:", lokasi_fitur_v2)

print("\nRingkasan hasil URL:")
display(
    data_hasil_url_v3.groupby(
        ["hasil_akhir", "kategori_risiko", "intelligence_status"]
    ).size().reset_index(name="jumlah_data")
)

print("\nRingkasan hasil file:")
display(
    data_hasil_file_v3.groupby(
        ["hasil_akhir_file_v3", "kategori_final_file_v3", "risiko_awal_file"]
    ).size().reset_index(name="jumlah_data")
)

print("\nContoh hasil URL:")
display(data_hasil_url_v3[kolom_tampilan_url_v3])

print("\nContoh hasil file:")
display(data_hasil_file_v3[kolom_tampilan_file_v3])

print("\nFile penting STEP 9:")
print(lokasi_engine_v3)
print(lokasi_hasil_url_v3)
print(lokasi_hasil_file_v3)
print(lokasi_hasil_url_dalam_file_v3)
print(lokasi_metadata_engine_v3)
print(lokasi_validasi_step9)

RINGKASAN STEP INI
Engine final: C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py
Model digunakan: C:\Users\ASUS\PHISHING\models\model_terbaik_intelligence_v2.pkl
Fitur digunakan: C:\Users\ASUS\PHISHING\reports\outputs\daftar_fitur_intelligence_v2.json

Ringkasan hasil URL:


,hasil_akhir,kategori_risiko,intelligence_status,jumlah_data
0,Berisiko,Sangat Tinggi,domain_mirip_brand,2
1,Berisiko,Sangat Tinggi,domain_mirip_brand_berisiko,1
2,Berisiko,Sangat Tinggi,kata_mencurigakan_tinggi,1
3,Berisiko,Sangat Tinggi,tiruan_brand_berisiko,6
4,Terlihat Aman,Rendah,resmi_terlihat_aman,5



Ringkasan hasil file:


,hasil_akhir_file_v3,kategori_final_file_v3,risiko_awal_file,jumlah_data
0,Berisiko,Sangat Tinggi,rendah,1
1,Berisiko,Sangat Tinggi,sedang,4
2,Berisiko,Sangat Tinggi,tinggi,1
3,Terlihat Aman,Rendah,rendah,1



Contoh hasil URL:


,url,domain,label_model,skor_model,skor_final,kategori_risiko,hasil_akhir,intelligence_status,official_brand,brand_detected,suspicious_keywords,lookalike_brand,lookalike_score,rekomendasi
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,Legitimate,20.40,20.40,Rendah,Terlihat Aman,resmi_terlihat_aman,Gunadarma,Gunadarma,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
1,https://baak.gunadarma.ac.id,baak.gunadarma.ac.id,Legitimate,31.60,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,Gunadarma,Gunadarma,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
2,https://www.bca.co.id,www.bca.co.id,Legitimate,4.05,4.05,Rendah,Terlihat Aman,resmi_terlihat_aman,BCA,BCA,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
3,https://www.shopee.co.id,www.shopee.co.id,Legitimate,9.22,9.22,Rendah,Terlihat Aman,resmi_terlihat_aman,Shopee,Shopee,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
4,https://www.microsoft.com,www.microsoft.com,Legitimate,48.00,24.00,Rendah,Terlihat Aman,resmi_terlihat_aman,Microsoft,Microsoft,,,0.0000,Alamat cocok dengan daftar domain resmi dan ti...
5,http://rricrosoft.com,rricrosoft.com,Phishing,99.60,99.60,Sangat Tinggi,Berisiko,domain_mirip_brand,,,,Microsoft,0.8421,Alamat terindikasi meniru brand atau domain re...
6,http://rnicrosoft.com,rnicrosoft.com,Phishing,99.60,99.60,Sangat Tinggi,Berisiko,domain_mirip_brand,,,,Microsoft,0.8421,Alamat terindikasi meniru brand atau domain re...
7,http://micros0ft-login-update.test,micros0ft-login-update.test,Phishing,98.80,98.80,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,,Microsoft,"login, update",Microsoft,1.0000,Alamat terindikasi meniru brand atau domain re...
8,http://bca-login-update.test,bca-login-update.test,Phishing,99.60,99.60,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,,BCA,"login, update",BCA,1.0000,Alamat terindikasi meniru brand atau domain re...
9,http://paypal-verify-account.test,paypal-verify-account.test,Phishing,100.00,100.00,Sangat Tinggi,Berisiko,tiruan_brand_berisiko,,PayPal,"account, verify",PayPal,1.0000,Alamat terindikasi meniru brand atau domain re...



Contoh hasil file:


,nama_file,ekstensi,risiko_awal_file,jumlah_url,jumlah_url_berisiko_v3,jumlah_url_perlu_tinjauan_v3,jumlah_kata_mencurigakan,jumlah_izin_apk_berisiko,skor_risiko_file,skor_final_file_v3,kategori_final_file_v3,hasil_akhir_file_v3,alasan_file,rekomendasi_final_file_v3
0,contoh_aplikasi_dummy.apk,.apk,tinggi,1,1,0,2,3,100,100,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
1,contoh_arsip_mencurigakan.zip,.zip,sedang,2,2,0,4,0,100,100,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
2,contoh_catatan_aman.txt,.txt,rendah,1,0,0,0,0,12,12,Rendah,Terlihat Aman,File mengandung URL.,File terlihat rendah risiko berdasarkan pemeri...
3,contoh_dokumen_link.docx,.docx,sedang,2,2,0,4,0,93,93,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
4,contoh_halaman_login.html,.html,sedang,1,1,0,5,0,74,80,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
5,contoh_pdf_link.pdf,.pdf,sedang,1,1,0,2,0,100,100,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...
6,contoh_pesan_mencurigakan.txt,.txt,rendah,1,1,0,5,0,55,80,Sangat Tinggi,Berisiko,File mengandung URL. Ada URL di dalam file yan...,File berisiko. Jangan dibuka atau dijalankan s...



File penting STEP 9:
C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_phishrisk_engine_v3_url.csv
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_phishrisk_engine_v3_file.csv
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_phishrisk_engine_v3_url_dalam_file.csv
C:\Users\ASUS\PHISHING\reports\outputs\metadata_phishrisk_engine_v3.json
C:\Users\ASUS\PHISHING\reports\outputs\validasi_step9_phishrisk_engine_v3.csv
